# v2 Curated Corpus — Validate & Export

Validates every entry in `curated_skeletons.py`, runs an eager-vs-inductor sanity check on each op, and writes `results/corpus_train.jsonl` if all entries pass.

**Run on Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

### Skeleton file format
```python
ENTRIES = [
    {
        "name": "elementwise_clamp_square",   # unique snake_case id, forms the origin
        "op_category": "elementwise_chain",   # must match OpCategory enum
        "input_shapes": [[1048576]],           # list of shapes, one per tensor arg
        "input_dtypes": ["float32"],           # float32 | float16 | bfloat16 | int64
        "code": """
def op(x):
    y = torch.clamp(x, -3.0, 3.0)
    return y * y + 0.25 * y
""",
    },
]
```

## Cell 1 — Locate skeleton file

In [ ]:
import importlib.util
import os
from pathlib import Path

SKELETON_FILENAME = "curated_skeletons.py"

# Candidate locations, checked in order
search_paths = [
    Path("."),
    Path("skeletons"),
    Path("scripts/v2Scraper/skeletons"),
    Path("../v2Scraper/skeletons"),
    Path("/content"),
    Path("/content/skeletons"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/tried/scripts/v2Scraper/skeletons"),
]

skeleton_path = None
for p in search_paths:
    candidate = p / SKELETON_FILENAME
    if candidate.exists():
        skeleton_path = candidate.resolve()
        break

if skeleton_path is None:
    raise FileNotFoundError(
        f"{SKELETON_FILENAME} not found. Upload it to Colab or mount Drive first.\n"
        f"Searched: {[str(p) for p in search_paths]}"
    )

print(f"Found: {skeleton_path}")

## Cell 2 — Load entries

In [ ]:
spec = importlib.util.spec_from_file_location("curated_skeletons", skeleton_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
ENTRIES = mod.ENTRIES

from collections import Counter
cats = Counter(e["op_category"] for e in ENTRIES)
print(f"Loaded {len(ENTRIES)} entries")
for cat, n in sorted(cats.items()):
    print(f"  {cat}: {n}")

## Cell 3 — Validation helpers

In [ ]:
import torch
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU — inductor sanity check will still run but is slower")

DTYPE_MAP = {
    "float32":  torch.float32,
    "float16":  torch.float16,
    "bfloat16": torch.bfloat16,
    "int64":    torch.int64,
    "int32":    torch.int32,
    "bool":     torch.bool,
}

# Mirrors the shared tolerance policy defaults (locked in docs/tolerance-policy.md)
def _tolerance(input_dtypes):
    if any(d in ("float16", "bfloat16") for d in input_dtypes):
        return {"atol": 1e-3, "rtol": 1e-3}
    return {"atol": 1e-5, "rtol": 1e-5}

def _make_tensors(input_shapes, input_dtypes, seed=42):
    torch.manual_seed(seed)
    tensors = []
    for shape, dtype_str in zip(input_shapes, input_dtypes):
        dtype = DTYPE_MAP[dtype_str]
        if dtype in (torch.int64, torch.int32):
            t = torch.randint(0, 100, shape, dtype=dtype, device=DEVICE)
        elif dtype == torch.bool:
            t = torch.randint(0, 2, shape, dtype=torch.bool, device=DEVICE)
        else:
            t = torch.randn(shape, dtype=dtype, device=DEVICE)
        tensors.append(t)
    return tensors

def _build_code(code_str):
    """Prepend standard imports and return exec namespace with op."""
    full = "import torch\nimport torch.nn.functional as F\n\n" + code_str.strip()
    ns = {}
    exec(full, ns)  # noqa: S102
    return full, ns["op"]

def validate_entry(entry):
    """Returns (ok: bool, message: str, full_code: str)."""
    name = entry["name"]

    # --- build op ---
    try:
        full_code, op = _build_code(entry["code"])
    except Exception as e:
        return False, f"exec failed: {e}", None

    # --- make inputs ---
    try:
        tensors = _make_tensors(entry["input_shapes"], entry["input_dtypes"])
    except Exception as e:
        return False, f"tensor creation failed: {e}", full_code

    # --- eager run ---
    try:
        with torch.no_grad():
            out_eager = op(*tensors)
        if not isinstance(out_eager, torch.Tensor):
            return False, f"op returned {type(out_eager).__name__}, expected Tensor", full_code
    except Exception as e:
        return False, f"eager run failed: {e}", full_code

    # --- inductor run ---
    try:
        compiled_op = torch.compile(op, backend="inductor", fullgraph=True)
        tensors2 = _make_tensors(entry["input_shapes"], entry["input_dtypes"])
        with torch.no_grad():
            out_inductor = compiled_op(*tensors2)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
    except Exception as e:
        return False, f"inductor failed: {e}", full_code

    # --- sanity check: eager vs inductor ---
    try:
        tol = _tolerance(entry["input_dtypes"])
        # integer/bool outputs: require exact match
        if out_eager.dtype in (torch.int64, torch.int32, torch.bool):
            if not out_eager.equal(out_inductor):
                return False, "eager/inductor exact mismatch (integer output)", full_code
        else:
            e32 = out_eager.float()
            i32 = out_inductor.float()
            if not torch.allclose(e32, i32, **tol, equal_nan=False):
                max_diff = (e32 - i32).abs().max().item()
                return False, f"eager/inductor mismatch: max_diff={max_diff:.3e} (tol={tol})", full_code
    except Exception as e:
        return False, f"sanity check error: {e}", full_code

    return True, "ok", full_code

print("Helpers ready.")

## Cell 4 — Run validation

In [ ]:
results = []
for i, entry in enumerate(ENTRIES):
    ok, msg, full_code = validate_entry(entry)
    results.append({"entry": entry, "ok": ok, "msg": msg, "full_code": full_code})
    status = "PASS" if ok else "FAIL"
    print(f"[{i+1:03d}/{len(ENTRIES)}] {status}  {entry['name']:<55}  {msg}")

n_pass = sum(r["ok"] for r in results)
n_fail = len(results) - n_pass
print(f"\n{'='*70}")
print(f"Passed: {n_pass}/{len(results)}    Failed: {n_fail}/{len(results)}")

## Cell 5 — Failure detail

In [ ]:
failures = [r for r in results if not r["ok"]]
if not failures:
    print("All entries passed — ready to export.")
else:
    print(f"{len(failures)} failure(s):\n")
    for r in failures:
        e = r["entry"]
        print(f"  name        : {e['name']}")
        print(f"  op_category : {e['op_category']}")
        print(f"  input_shapes: {e['input_shapes']}")
        print(f"  input_dtypes: {e['input_dtypes']}")
        print(f"  error       : {r['msg']}")
        print(f"  code        :\n{e['code']}")
        print("-" * 60)

## Cell 6 — Export JSONL

Only runs if all entries passed. Writes `results/corpus_train.jsonl` next to this notebook.

In [ ]:
import json
import uuid

if n_fail > 0:
    print(f"Blocked: {n_fail} entry/entries failed. Fix them and re-run validation first.")
else:
    # Deterministic UUIDv5 — stable across reruns for the same content
    _NS = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")  # uuid.NAMESPACE_URL

    def _make_id(origin, code, shapes, dtypes):
        stable = f"{origin}|{code.strip()}|{shapes}|{dtypes}"
        return str(uuid.uuid5(_NS, stable))

    output_rows = []
    for r in results:
        e = r["entry"]
        origin = f"curated/{e['name']}"
        pytorch_code = r["full_code"] + "\n"
        row = {
            "difficulty":      None,
            "example_id":      _make_id(origin, e["code"], e["input_shapes"], e["input_dtypes"]),
            "input_dtypes":    e["input_dtypes"],
            "input_shapes":    e["input_shapes"],
            "op_category":     e["op_category"],
            "origin":          origin,
            "pytorch_code":    pytorch_code,
            "rng_seed":        42,
            "split":           "train",
        }
        output_rows.append(row)

    out_dir = Path(skeleton_path).parent / "results"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / "corpus_train.jsonl"

    with open(out_path, "w") as f:
        for row in output_rows:
            f.write(json.dumps(row) + "\n")

    print(f"Wrote {len(output_rows)} rows → {out_path}")
    cats = Counter(r["op_category"] for r in output_rows)
    for cat, n in sorted(cats.items()):
        print(f"  {cat}: {n}")